# Лабораторная работа: Бинарная классификация факта ДТП на основе данных о дорожном трафике

**Курс:** Машинное обучение и анализ данных  

---

## Источник данных

Датасет: **[Synthetic Traffic Benchmark](https://www.kaggle.com/datasets/razanihababdellatif/emergency-dispatch-and-traffic-physics-dataset)** (синтетический бенчмарк дорожного трафика)  
Платформа: [Kaggle](https://www.kaggle.com/)  

> Датасет содержит ~20 000 записей о дорожных событиях, включающих информацию о времени, типе дороги, погодных условиях, видимости, характеристиках водителя и транспортного средства, а также сведения о произошедших ДТП (факт аварии, тип столкновения, тяжесть, время реагирования и т.д.).

In [1]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns

# Предобработка данных
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Модели
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Метрики
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Настройки отображения
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42

## 0. Постановка задачи

### Описание предметной области

В датасете собраны наблюдения за дорожным трафиком: для каждой записи известны контекстные условия (время суток, день недели, праздник), характеристики дороги (тип зоны, тип дороги, тип перекрёстка, ограничение скорости), погодные и визуальные условия (погода, состояние покрытия, видимость), параметры водителя и автомобиля (тип ТС, возраст водителя, стаж, возраст автомобиля), а также текущая скорость и индекс загруженности. Для части записей зафиксирован факт ДТП и его последствия.

### Формулировка задачи

**Тип задачи:** бинарная классификация.

**Целевая переменная:** `Accident_Occurred` ∈ {0, 1}
- `0` — ДТП не произошло;
- `1` — ДТП произошло.

**Признаки (предикторы):** все переменные, известные **в момент наблюдения до события**:
- временные: `Hour`, `Month`, `Day_of_Week`, `Time_of_Day`, `Is_Holiday`;
- дорожные: `Area_Type`, `Road_Type`, `Intersection_Type`, `Speed_Limit_kmh`;
- условия: `Weather`, `Road_Condition`, `Visibility_m`, `Congestion_Index`;
- водитель/ТС: `Vehicle_Type`, `Driver_Age`, `Driver_Experience_yrs`, `Vehicle_Age_yrs`;
- текущая динамика: `Recorded_Speed_kmh`.

**Исключаемые признаки (утечка данных / пост-фактум информация):**
`Collision_Type`, `Vehicles_Involved`, `Accident_Severity`, `Dispatch_Priority`, `Actual_Response_Time_min`, `Road_Closure_Duration_min`, `Baseline_Expected_Response_Time_min`, `Record_ID`, `Timestamp`, `Date`.

Причина: эти признаки становятся известны только *после* аварии и/или являются её прямыми следствиями — их использование привело бы к «протеканию» целевой переменной в признаки и завышенным, но бесполезным на практике метрикам.

### Прикладная цель

Построить модель, которая по условиям движения и характеристикам участника предсказывает вероятность ДТП. Это позволит:
- выявлять «опасные» комбинации условий (погода × видимость × скорость × тип дороги);
- приоритизировать профилактические меры и маршруты патрулирования;
- сравнить несколько классификаторов и подобрать оптимальную сложность модели (число соседей в kNN).

### Метрики качества

Из-за ожидаемой несбалансированности классов (ДТП — редкое событие) основной метрикой выбираем **F1-score** для положительного класса и **ROC-AUC**, вспомогательно — accuracy и матрицу рассогласования. Разбиение train/test — стратифицированное (`stratify=y`), чтобы сохранить долю классов.

### План работы

1. Загрузка и первичный осмотр данных.
2. Разбиение на обучающую и тестовую выборки.
3. Разведочный анализ (EDA): описательные статистики, распределения, корреляционная матрица, интерпретация.
4. Обработка выбросов.
5. Обработка пропущенных значений.
6. Кодирование категориальных признаков.
7. Нормализация числовых признаков (обязательна для kNN).
8. Обучение kNN-классификатора (обоснование выбора метода).
9. Оценка на train/test, матрица рассогласования, подбор гиперпараметра `k`.
10. Сравнение с другими классификаторами (LogisticRegression, RandomForest).
11. Борьба с несбалансированностью классов.
12. Отбор признаков / удаление коррелированных переменных.
13. Выводы.

## 1. Загрузка данных

In [2]:
url = "https://raw.githubusercontent.com/Svetlana-plv/Machine_Learning/refs/heads/main/synthetic_traffic_benchmark_200k.csv"
data_raw = pd.read_csv(url)
print(f"Размер датасета: {data_raw.shape[0]} строк, {data_raw.shape[1]} столбцов")

Размер датасета: 200000 строк, 29 столбцов


### 1.1. Проверка корректности чтения данных

Прежде чем приступать к анализу, необходимо убедиться, что датасет прочитан полностью и без искажений.

In [3]:
# Проверка корректности чтения
print("Тип объекта:", type(data_raw))
print("Размерность (строки, столбцы):", data_raw.shape)

Тип объекта: <class 'pandas.core.frame.DataFrame'>
Размерность (строки, столбцы): (200000, 29)


In [4]:
# Первые строки таблицы
display(data_raw.head())

# Случайная выборка — чтобы убедиться, что данные не отсортированы и читаются целиком
display(data_raw.sample(5, random_state=RANDOM_STATE))

,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,Intersection_Type,Speed_Limit_kmh,Weather,Road_Condition,Visibility_m,Congestion_Index,Vehicle_Type,Driver_Age,Driver_Experience_yrs,Vehicle_Age_yrs,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min
0,REC_0000000,2024-03-14 17:35:06.841829282,2024-03-14,17,3,Thursday,Evening Peak,0,Urban,Interstate,NaN,90,Clear,Dry,10905.0,100.0,SUV,40.0,16.0,7.2,49.0,6.2,0,NaN,0,No Accident,NaN,0.0,0.0
1,REC_0000001,2023-05-25 08:10:56.398060108,2023-05-25,8,5,Thursday,Morning Peak,0,Suburban,Interstate,NaN,100,Clear,Dry,12220.6,100.0,Sedan,48.0,26.0,9.4,70.1,15.0,0,NaN,0,No Accident,NaN,0.0,0.0
2,REC_0000002,2023-11-07 08:25:17.721608262,2023-11-07,8,11,Tuesday,Morning Peak,0,Suburban,Interstate,NaN,100,Clear,Dry,11302.4,100.0,Sedan,37.0,13.3,2.9,71.9,14.6,0,NaN,0,No Accident,NaN,0.0,0.0
3,REC_0000003,2024-07-24 09:30:00.947374674,2024-07-24,9,7,Wednesday,Morning Peak,0,Suburban,Collector,NaN,50,Clear,Dry,8641.9,50.0,SUV,48.0,12.2,4.4,58.7,13.2,0,NaN,0,No Accident,NaN,0.0,0.0
4,REC_0000004,2024-04-03 09:34:07.316651267,2024-04-03,9,4,Wednesday,Morning Peak,0,Urban,Local,T-Junction,30,Rain,Wet,3072.2,78.2,SUV,32.0,12.4,1.8,25.4,7.8,0,NaN,0,No Accident,NaN,0.0,0.0


,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,Intersection_Type,Speed_Limit_kmh,Weather,Road_Condition,Visibility_m,Congestion_Index,Vehicle_Type,Driver_Age,Driver_Experience_yrs,Vehicle_Age_yrs,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min
119737,REC_0119737,2024-03-26 01:22:38.583062081,2024-03-26,1,3,Tuesday,Night,0,Rural,Interstate,NaN,120,Rain,Wet,1984.1,27.2,SUV,28.0,0.4,1.6,110.7,21.3,0,NaN,0,No Accident,NaN,0.0,0.0
72272,REC_0072272,2025-01-23 18:44:13.669805807,2025-01-23,18,1,Thursday,Evening Peak,0,Rural,Arterial,4-Way,80,Clear,Dry,9021.6,39.9,SUV,16.0,0.0,22.4,95.1,23.5,0,NaN,0,No Accident,NaN,0.0,0.0
158154,REC_0158154,2024-01-02 19:39:02.727484095,2024-01-02,19,1,Tuesday,Evening Peak,0,Suburban,Collector,T-Junction,50,Clear,Dry,11414.0,63.4,SUV,35.0,4.3,1.3,55.1,14.0,0,NaN,0,No Accident,NaN,0.0,0.0
65426,REC_0065426,2025-02-26 07:51:25.145217268,2025-02-26,7,2,Wednesday,Morning Peak,0,Suburban,Local,T-Junction,30,Fog,Dry,223.5,40.0,Sedan,36.0,9.8,9.0,24.2,11.9,1,Side-Impact,2,Moderate,Medium,24.1,189.3
30074,REC_0030074,2025-05-18 18:07:08.972592121,2025-05-18,18,5,Sunday,Evening Peak,0,Urban,Collector,4-Way,40,Clear,Dry,8947.0,57.7,Sedan,39.0,23.0,5.0,46.3,8.8,0,NaN,0,No Accident,NaN,0.0,0.0


In [5]:
# Проверка на полностью пустые строки/столбцы
print("Полностью пустых строк:", data_raw.isna().all(axis=1).sum())
print("Полностью пустых столбцов:", data_raw.isna().all(axis=0).sum())

# Проверка типов
print("\nТипы данных по столбцам:")
print(data_raw.dtypes.value_counts())

Полностью пустых строк: 0
Полностью пустых столбцов: 0

Типы данных по столбцам:
object     14
float64     9
int64       6
Name: count, dtype: int64


### 1.2. Сводка по каждому столбцу

In [6]:
data_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 29 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Record_ID                            200000 non-null  object 
 1   Timestamp                            200000 non-null  object 
 2   Date                                 200000 non-null  object 
 3   Hour                                 200000 non-null  int64  
 4   Month                                200000 non-null  int64  
 5   Day_of_Week                          200000 non-null  object 
 6   Time_of_Day                          200000 non-null  object 
 7   Is_Holiday                           200000 non-null  int64  
 8   Area_Type                            200000 non-null  object 
 9   Road_Type                            200000 non-null  object 
 10  Intersection_Type                    131481 non-null  object 
 11  Speed_Limit_k


**Описание столбцов:**

- `Record_ID` — уникальный идентификатор записи
- `Timestamp` — полная дата-время события
- `Date` — дата события
- `Hour` — час суток (0–23)
- `Month` — месяц года (1–12)
- `Day_of_Week` — день недели
- `Time_of_Day` — обобщённая часть суток (Morning/Afternoon/Evening/Night)
- `Is_Holiday` — флаг праздничного дня (0/1)
- `Area_Type` — тип зоны (Urban / Rural / Highway)
- `Road_Type` — тип дороги
- `Intersection_Type` — тип перекрёстка
- `Speed_Limit_kmh` — разрешённое ограничение скорости, км/ч
- `Weather` — погодные условия
- `Road_Condition` — состояние дорожного покрытия
- `Visibility_m` — видимость в метрах
- `Congestion_Index` — индекс загруженности дороги
- `Vehicle_Type` — тип транспортного средства
- `Driver_Age` — возраст водителя, лет
- `Driver_Experience_yrs` — стаж вождения, лет
- `Vehicle_Age_yrs` — возраст автомобиля, лет
- `Recorded_Speed_kmh` — фактически зафиксированная скорость, км/ч
- `Baseline_Expected_Response_Time_min` — ожидаемое время реакции служб (пост-фактум)
- `Accident_Occurred` — факт ДТП (0 — не произошло, 1 — произошло) 
- `Collision_Type` — тип столкновения (пост-фактум)
- `Vehicles_Involved` — число участников ДТП (пост-фактум)
- `Accident_Severity` — тяжесть ДТП (пост-фактум)
- `Dispatch_Priority` — приоритет диспетчера (пост-фактум)
- `Actual_Response_Time_min` — фактическое время реакции служб (пост-фактум)
- `Road_Closure_Duration_min` — длительность перекрытия дороги (пост-фактум)

### 1.3. Проверка наличия дублирующихся строк


In [14]:
counts = data_raw['Record_ID'].value_counts()
print(counts[counts > 1])


Record_ID
REC_0002958    2
REC_0086356    2
REC_0161587    2
REC_0145781    2
REC_0193705    2
              ..
REC_0184830    2
REC_0152045    2
REC_0049820    2
REC_0118300    2
REC_0143052    2
Name: count, Length: 597, dtype: int64


In [16]:
# Дубли по всем столбцам, кроме Record_ID
cols = [c for c in data_raw.columns if c != 'Record_ID']
full_dups = data_raw.duplicated(subset=cols, keep=False)
print("Полных дублей (без учёта ID):", full_dups.sum())

# Посмотреть на примеры
display(data_raw[full_dups].sort_values(cols).head(10))

# Дубли только по Record_ID
id_dups = data_raw['Record_ID'].duplicated(keep=False)
print("Строк с повторяющимся Record_ID:", id_dups.sum())

# Пересечение: и ID повторяется, и строка полностью совпадает
print("Совпадает и ID, и все признаки:", (full_dups & id_dups).sum())


Полных дублей (без учёта ID): 1194


,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,Intersection_Type,Speed_Limit_kmh,Weather,Road_Condition,Visibility_m,Congestion_Index,Vehicle_Type,Driver_Age,Driver_Experience_yrs,Vehicle_Age_yrs,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min
91211,REC_0091211,2022-01-10 17:48:57.501033079,2022-01-10,17,1,Monday,Evening Peak,0,Rural,Interstate,NaN,120,Snow,Icy,1071.4,99.2,SUV,19.0,3.0,7.3,54.7,25.0,0,NaN,0,No Accident,NaN,0.0,0.0
199141,REC_0091211,2022-01-10 17:48:57.501033079,2022-01-10,17,1,Monday,Evening Peak,0,Rural,Interstate,NaN,120,Snow,Icy,1071.4,99.2,SUV,19.0,3.0,7.3,54.7,25.0,0,NaN,0,No Accident,NaN,0.0,0.0
109420,REC_0109420,2022-01-17 06:23:35.853323670,2022-01-17,6,1,Monday,Morning Peak,1,Rural,Arterial,4-Way,80,Fog,Wet,303.6,30.9,Truck,34.0,17.0,7.2,71.1,21.8,0,NaN,0,No Accident,NaN,0.0,0.0
199401,REC_0109420,2022-01-17 06:23:35.853323670,2022-01-17,6,1,Monday,Morning Peak,1,Rural,Arterial,4-Way,80,Fog,Wet,303.6,30.9,Truck,34.0,17.0,7.2,71.1,21.8,0,NaN,0,No Accident,NaN,0.0,0.0
90202,REC_0090202,2022-01-20 17:49:01.957910330,2022-01-20,17,1,Thursday,Evening Peak,0,Suburban,Arterial,Roundabout,60,Clear,Dry,9106.5,74.7,SUV,51.0,35.0,12.8,67.8,9.6,0,NaN,0,No Accident,NaN,0.0,0.0
199305,REC_0090202,2022-01-20 17:49:01.957910330,2022-01-20,17,1,Thursday,Evening Peak,0,Suburban,Arterial,Roundabout,60,Clear,Dry,9106.5,74.7,SUV,51.0,35.0,12.8,67.8,9.6,0,NaN,0,No Accident,NaN,0.0,0.0
29961,REC_0029961,2022-01-29 16:35:03.002375130,2022-01-29,16,1,Saturday,Evening Peak,0,Urban,Arterial,4-Way,50,Clear,Dry,6611.5,84.7,Sedan,45.0,5.2,14.3,36.9,9.0,0,NaN,0,No Accident,NaN,0.0,0.0
199590,REC_0029961,2022-01-29 16:35:03.002375130,2022-01-29,16,1,Saturday,Evening Peak,0,Urban,Arterial,4-Way,50,Clear,Dry,6611.5,84.7,Sedan,45.0,5.2,14.3,36.9,9.0,0,NaN,0,No Accident,NaN,0.0,0.0
74118,REC_0074118,2022-02-01 07:38:31.558701408,2022-02-01,7,2,Tuesday,Morning Peak,0,Suburban,Arterial,T-Junction,60,Fog,Wet,265.6,81.1,Sedan,35.0,19.0,8.9,56.1,12.8,0,NaN,0,No Accident,NaN,0.0,0.0
199257,REC_0074118,2022-02-01 07:38:31.558701408,2022-02-01,7,2,Tuesday,Morning Peak,0,Suburban,Arterial,T-Junction,60,Fog,Wet,265.6,81.1,Sedan,35.0,19.0,8.9,56.1,12.8,0,NaN,0,No Accident,NaN,0.0,0.0


Строк с повторяющимся Record_ID: 1194
Совпадает и ID, и все признаки: 1194


Избавимся от дубликатов и оставим по одному экземпляру каждого:

In [17]:
before = len(data_raw)
data_raw = data_raw.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Удалено дублей: {before - len(data_raw)}")
print(f"Размер после удаления: {data_raw.shape}")

Удалено дублей: 597
Размер после удаления: (199403, 29)


In [18]:
# Проверяем, что дублей не осталось
print("Дублей по Record_ID:", data_raw['Record_ID'].duplicated().sum())
print("Полных дублей:", data_raw.duplicated().sum())

# Убеждаемся, что Record_ID теперь уникален
assert data_raw['Record_ID'].is_unique, "Остались дубли Record_ID!"
print("Record_ID уникален ✓")

Дублей по Record_ID: 0
Полных дублей: 0
Record_ID уникален ✓


**Обработка дубликатов:** В датасете обнаружено 597 пар полностью идентичных строк (1194 записи): совпадают и Record_ID, и все 28 остальных признаков. Это технические дубли, не несущие новой информации. Они удалены методом drop_duplicates(keep='first'), так как при случайном разбиении на train/test один и тот же объект мог попасть в обе выборки, что привело бы к утечке данных и завышенным метрикам качества. После удаления осталось 199 403 уникальные записи.

### 1.4. Выбор индекса таблицы

Теперь в полученном списке нет повторяющихся записей. В качестве индекса выбираем столбец `Record_ID` — уникальный идентификатор записи. Используем его
как индекс DataFrame — это удобно для идентификации строк и не влияет на
моделирование, поскольку `Record_ID` исключён из списка признаков.

In [19]:
data_raw = data_raw.set_index('Record_ID')
print("Индекс установлен:", data_raw.index.name)
print("Уникален:", data_raw.index.is_unique)

Индекс установлен: Record_ID
Уникален: True


### 1.5. Построение целевой переменной

Посмотрим на распределение целевой переменной (`Accident_Occurred`), чтобы оценить баланс классов:

In [20]:
data_raw['Accident_Occurred'].describe()

count    199403.000000
mean          0.041188
std           0.198725
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: Accident_Occurred, dtype: float64

In [22]:
print('Записей без ДТП (0): ', (data_raw['Accident_Occurred'] == 0).sum())
print('Записей с ДТП (1):   ', (data_raw['Accident_Occurred'] == 1).sum())

Записей без ДТП (0):  191190
Записей с ДТП (1):    8213


Датасет содержит подавляющее большинство записей без ДТП (класс 0) и лишь малую долю записей с произошедшими авариями (класс 1). Данные сильно несбалансированы. 

In [24]:
# Формируем целевую переменную (target)
y = data_raw['Accident_Occurred'].copy()
# Доли классов
data_raw['Accident_Occurred'].value_counts(normalize=True)

Accident_Occurred
0    0.958812
1    0.041188
Name: proportion, dtype: float64

### 1.6. Исключение лишних столбцов

Часть признаков напрямую связана с тем, что предсказывает модель (пост-фактум информация), либо не несёт полезной информации для машинного обучения. Такие столбцы необходимо исключить, чтобы избежать утечки данных (data leakage) и не завышать метрики качества.

**Утечка данных (стали известны после ДТП):**
- `Collision_Type`, `Accident_Severity` — следствия аварии;
- `Vehicles_Involved` — количество участников;
- `Dispatch_Priority`, `Actual_Response_Time_min`, `Baseline_Expected_Response_Time_min`, `Road_Closure_Duration_min` — параметры работы экстренных служб.

**Технические / служебные признаки (не несут физического смысла для модели):**
- `Timestamp`, `Date` — дублируют информацию о времени из `Hour`, `Month`, `Day_of_Week`;
- `Record_ID` — является индексом (уже убран из таблицы как столбец).


In [26]:
# 1. Признаки, являющиеся следствием аварии (утечка)
leak_columns = [
    'Collision_Type', 'Vehicles_Involved', 'Accident_Severity', 
    'Dispatch_Priority', 'Actual_Response_Time_min', 
    'Road_Closure_Duration_min', 'Baseline_Expected_Response_Time_min'
]

# 2. Технические признаки (служебные)
technical_columns = ['Timestamp', 'Date']

# Объединяем списки для удаления
columns_to_drop = leak_columns + technical_columns

# Удаляем их из датафрейма
data_clean = data_raw.drop(columns=columns_to_drop)

# Проверяем результат
print(f"Удалено столбцов: {len(columns_to_drop)}")
print(f"Размерность после удаления: {data_clean.shape}")
print("\nОставшиеся столбцы:")
print(data_clean.columns.tolist())

Удалено столбцов: 9
Размерность после удаления: (199403, 19)

Оставшиеся столбцы:
['Hour', 'Month', 'Day_of_Week', 'Time_of_Day', 'Is_Holiday', 'Area_Type', 'Road_Type', 'Intersection_Type', 'Speed_Limit_kmh', 'Weather', 'Road_Condition', 'Visibility_m', 'Congestion_Index', 'Vehicle_Type', 'Driver_Age', 'Driver_Experience_yrs', 'Vehicle_Age_yrs', 'Recorded_Speed_kmh', 'Accident_Occurred']


## 2. Разбиение на обучающую и тестовую выборки

Сформируем матрицу признаков `X`, исключив целевую переменную `Accident_Occurred`, и вектор ответов `y`.

Разбиение выполним со стратификацией (`stratify=y`), чтобы сохранить исходную пропорцию классов (~4.1% ДТП) в обеих выборках. Тестовая выборка составит 20% от всех данных.

In [27]:
# 1. Формируем X (признаки) и y (целевая переменная) из очищенных данных
X = data_clean.drop(columns=['Accident_Occurred'])
y = data_clean['Accident_Occurred']

# 2. Разбиение на train/test (20% на тест)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=RANDOM_STATE
)

Проверим размерности полученных выборок и убедимся, что доли классов сохранились.

In [28]:
print(f"Размер X_train: {X_train.shape}")
print(f"Размер X_test:  {X_test.shape}")
print(f"\nДоля ДТП в train: {y_train.mean():.4f}")
print(f"Доля ДТП в test:  {y_test.mean():.4f}")

# Проверка, что признаки и таргет согласованы по индексам
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
print("\nИндексы X и y согласованы ✓")

Размер X_train: (159522, 18)
Размер X_test:  (39881, 18)

Доля ДТП в train: 0.0412
Доля ДТП в test:  0.0412

Индексы X и y согласованы ✓
